# Analisis & Normalisasi Dataset Mobile Legends (MLBB)

Notebook ini bertujuan untuk melakukan **import data** dari semua dataset game (Game 1 - 10) di folder `dataset/` dan memprosesnya menjadi **2 versi normalisasi**:
1. **Versi Global/Total Match (`_total`)**: Normalisasi kolom `Hero Damage`, `Turret Damage`, dan `Damage Taken` sebagai persentase dari total seluruh data (10 pemain dalam satu game).
2. **Versi Per-Team (`_team`)**: Normalisasi untuk pemain 1-5 (Team 1) dan pemain 6-10 (Team 2) secara terpisah sebagai persentase dari total team masing-masing.

Hasil pemrosesan akan disimpan kembali ke folder `dataset/` dengan format penamaan:
- `game{X}_total.csv` untuk normalisasi global seluruh match.
- `game{X}_team.csv` untuk normalisasi per team.

In [101]:
!pip install pandas
!pip install matplotlib
!pip install seaborn


[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [102]:
import os
import re
import glob
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split

print("Libraries successfully imported!")

Libraries successfully imported!


## 1. Import dan Urutkan Dataset Game

Kita akan memindai folder `dataset/` untuk mencari file CSV game, mengekstrak nomor gamenya menggunakan regex, lalu mengurutkannya secara numerik.

In [103]:
dataset_dir = "dataset"

# Cari semua file CSV yang mengandung kata 'game'
csv_files = glob.glob(os.path.join(dataset_dir, "*[gG]ame*.csv"))

# Fungsi helper untuk ekstraksi nomor game secara aman
def extract_game_num(filename):
    match = re.search(r'[gG]ame\s*(\d+)', os.path.basename(filename))
    return int(match.group(1)) if match else 999

# Urutkan file berdasarkan nomor game
csv_files = sorted(list(set(csv_files)), key=extract_game_num)

print(f"Menemukan {len(csv_files)} file dataset:")
for f in csv_files:
    print(f" - {os.path.basename(f)} (Game {extract_game_num(f)})")

Menemukan 10 file dataset:
 - Input Gambar Tubes Alin Metnum MLBB (Responses) - game1.csv (Game 1)
 - Input Gambar Tubes Alin Metnum MLBB (Responses) - game2.csv (Game 2)
 - Input Gambar Tubes Alin Metnum MLBB (Responses) - game3.csv (Game 3)
 - Input Gambar Tubes Alin Metnum MLBB (Responses) - game4.csv (Game 4)
 - Input Gambar Tubes Alin Metnum MLBB (Responses) - Game5.csv (Game 5)
 - Input Gambar Tubes Alin Metnum MLBB (Responses) - game6.csv (Game 6)
 - Input Gambar Tubes Alin Metnum MLBB (Responses) - game 7.csv (Game 7)
 - Input Gambar Tubes Alin Metnum MLBB (Responses) - game 8.csv (Game 8)
 - Input Gambar Tubes Alin Metnum MLBB (Responses) - game 9.csv (Game 9)
 - Input Gambar Tubes Alin Metnum MLBB (Responses) - game 10.csv (Game 10)


## 2. Pemrosesan Versi 1: Normalisasi Total Match (Global)

Pada versi ini, kolom `Hero Damage`, `Turret Damage`, dan `Damage Taken` dikonversi menjadi persentase (%) terhadap **total keseluruhan match** (jumlah nilai dari ke-10 pemain).

In [104]:
def load_and_clean_df(file_path, game_num=None):
    """Load a CSV file and perform basic cleaning/typing so downstream code can assume consistent columns."""
    df = pd.read_csv(file_path)
    # Trim whitespace from column names
    df.columns = [c.strip() for c in df.columns]
    # Ensure numeric columns are numeric and fill missing with 0
    numeric_cols = ['K','D','A','Hero Damage','Turret Damage','Damage Taken','Teamfight Participation','Skor']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    # Keep only the first 10 rows (players 1-10) and reset index
    if len(df) < 10:
        raise ValueError(f"File {os.path.basename(file_path)} has less than 10 rows ({len(df)}).")
    df = df.iloc[:10].reset_index(drop=True)
    return df

cols_to_normalize = ["Hero Damage", "Turret Damage", "Damage Taken"]
# Simpan per-game hasil normalisasi sebagai DataFrame di-memory
per_game_total = {}
print("=== Memulai Normalisasi Total Match (in-memory) ===")
for file_path in csv_files:
    game_num = extract_game_num(file_path)
    if game_num == 999:
        continue
    df = load_and_clean_df(file_path, game_num)
    df_total = df.copy()

    # Hitung persentase dari total match
    for col in cols_to_normalize:
        total_sum = df_total[col].sum()
        if total_sum > 0:
            df_total[col] = (df_total[col] / total_sum * 100).round(2)
        else:
            df_total[col] = 0.0

    # Rename kolom agar lebih informatif
    df_total.rename(columns={
        "Hero Damage": "Hero Dmg (%)",
        "Turret Damage": "Turret Dmg (%)",
        "Damage Taken": "Dmg Taken (%)",
        "Teamfight Participation": "TFP (%)"
    }, inplace=True)

    # Simpan ke dict in-memory
    per_game_total[game_num] = df_total
    print(f"Game {game_num} -> Stored in `per_game_total[{game_num}]` (in-memory)")

=== Memulai Normalisasi Total Match (in-memory) ===
Game 1 -> Stored in `per_game_total[1]` (in-memory)
Game 2 -> Stored in `per_game_total[2]` (in-memory)
Game 3 -> Stored in `per_game_total[3]` (in-memory)
Game 4 -> Stored in `per_game_total[4]` (in-memory)
Game 5 -> Stored in `per_game_total[5]` (in-memory)
Game 6 -> Stored in `per_game_total[6]` (in-memory)
Game 7 -> Stored in `per_game_total[7]` (in-memory)
Game 8 -> Stored in `per_game_total[8]` (in-memory)
Game 9 -> Stored in `per_game_total[9]` (in-memory)
Game 10 -> Stored in `per_game_total[10]` (in-memory)


## 3. Pemrosesan Versi 2: Normalisasi Per-Team

Pada versi ini, data dibagi menjadi:
- **Team 1 (Ally)**: Baris 1-5 (indeks 0-4)
- **Team 2 (Enemy)**: Baris 6-10 (indeks 5-9)

Normalisasi kolom `Hero Damage`, `Turret Damage`, dan `Damage Taken` dihitung berdasarkan total dari masing-masing team saja.

In [105]:
print("=== Memulai Normalisasi Per Team (in-memory) ===")
per_game_team = {}
for file_path in csv_files:
    game_num = extract_game_num(file_path)
    if game_num == 999:
        continue
    df = load_and_clean_df(file_path, game_num)

    # Pisahkan per team
    team1 = df.iloc[0:5].copy()
    team2 = df.iloc[5:10].copy()

    for col in cols_to_normalize:
        # Normalisasi Team 1
        t1_sum = team1[col].sum()
        if t1_sum > 0:
            team1[col] = (team1[col] / t1_sum * 100).round(2)
        else:
            team1[col] = 0.0

        # Normalisasi Team 2
        t2_sum = team2[col].sum()
        if t2_sum > 0:
            team2[col] = (team2[col] / t2_sum * 100).round(2)
        else:
            team2[col] = 0.0

    # Gabungkan kembali
    df_team_normalized = pd.concat([team1, team2])

    # Rename kolom
    df_team_normalized.rename(columns={
        "Hero Damage": "Hero Dmg (%)",
        "Turret Damage": "Turret Dmg (%)",
        "Damage Taken": "Dmg Taken (%)",
        "Teamfight Participation": "TFP (%)"
    }, inplace=True)

    # Simpan ke dict in-memory
    per_game_team[game_num] = df_team_normalized
    print(f"Game {game_num} -> Stored in `per_game_team[{game_num}]` (in-memory)")

=== Memulai Normalisasi Per Team (in-memory) ===
Game 1 -> Stored in `per_game_team[1]` (in-memory)
Game 2 -> Stored in `per_game_team[2]` (in-memory)
Game 3 -> Stored in `per_game_team[3]` (in-memory)
Game 4 -> Stored in `per_game_team[4]` (in-memory)
Game 5 -> Stored in `per_game_team[5]` (in-memory)
Game 6 -> Stored in `per_game_team[6]` (in-memory)
Game 7 -> Stored in `per_game_team[7]` (in-memory)
Game 8 -> Stored in `per_game_team[8]` (in-memory)
Game 9 -> Stored in `per_game_team[9]` (in-memory)
Game 10 -> Stored in `per_game_team[10]` (in-memory)


In [106]:
# Preview per-game DataFrames (prefer in-memory dicts)
if 'per_game_total' in globals() and 1 in per_game_total:
    print('=== PREVIEW per_game_total[1] (Normalisasi Match) ===')
    display(per_game_total[1].head(5))
else:
    try:
        df_t_preview = pd.read_csv(os.path.join(dataset_dir, "game1_total.csv"))
        print('Preview loaded from game1_total.csv')
        display(df_t_preview.head(5))
    except Exception:
        print('No per-game total preview available')

if 'per_game_team' in globals() and 1 in per_game_team:
    print('=== PREVIEW per_game_team[1] (Normalisasi Team) ===')
    display(per_game_team[1].head(5))
else:
    try:
        df_tm_preview = pd.read_csv(os.path.join(dataset_dir, "game1_team.csv"))
        print('Preview loaded from game1_team.csv')
        display(df_tm_preview.head(5))
    except Exception:
        print('No per-game team preview available')

=== PREVIEW per_game_total[1] (Normalisasi Match) ===


,K,D,A,Hero Dmg (%),Turret Dmg (%),Dmg Taken (%),TFP (%),Skor
0,7,5,5,8.26,33.36,9.71,52,7.8
1,2,6,6,7.19,9.14,4.57,35,4.8
2,0,4,12,4.63,3.63,7.29,52,6.9
3,7,7,4,9.73,29.62,35.31,48,6.0
4,10,5,5,11.08,2.91,13.99,65,9.3


=== PREVIEW per_game_team[1] (Normalisasi Team) ===


,K,D,A,Hero Dmg (%),Turret Dmg (%),Dmg Taken (%),TFP (%),Skor
0,7,5,5,20.19,42.40,13.70,52,7.8
1,2,6,6,17.58,11.62,6.45,35,4.8
2,0,4,12,11.31,4.62,10.29,52,6.9
3,7,7,4,23.80,37.66,49.82,48,6.0
4,10,5,5,27.11,3.70,19.74,65,9.3


## 4. Menggabungkan semua data ke satu DF besar yang terbagi menjadi dua

Menggabungkan semua data sesuai dengan jenis normalisasinya

In [107]:
print("=== Menggabungkan Semua file per-game ke DataFrame (in-memory) ===")
# Jika per-game dicts tersedia, gunakan mereka first
df_all_total = None
if 'per_game_total' in globals() and per_game_total:
    # concat dan pastikan index di-reset tanpa menambahkan kolom index baru
    df_all_total = pd.concat(list(per_game_total.values()), ignore_index=True).reset_index(drop=True)
    print(f'Created df_all_total from in-memory per_game_total ({len(df_all_total)} rows)')
else:
    total_files = sorted(glob.glob(os.path.join(dataset_dir, "game*_total.csv")), key=extract_game_num)
    if total_files:
        dfs = []
        for f in total_files:
            g = extract_game_num(f)
            # baca tanpa menggunakan kolom index yang tersimpan
            d = pd.read_csv(f, index_col=False)
            # hapus kolom yang bernama 'Unnamed' (index tersimpan) jika ada
            d = d.loc[:, ~d.columns.str.contains('^Unnamed')]
            dfs.append(d)
        if dfs:
            df_all_total = pd.concat(dfs, ignore_index=True).reset_index(drop=True)
            print(f'Created df_all_total from CSV files ({len(df_all_total)} rows)')
        else:
            print('No valid total CSVs to aggregate.')
    else:
        print('No per-game total data available to aggregate.')

# Team aggregation (prefer in-memory)
df_all_team = None
if 'per_game_team' in globals() and per_game_team:
    df_all_team = pd.concat(list(per_game_team.values()), ignore_index=True).reset_index(drop=True)
    print(f'Created df_all_team from in-memory per_game_team ({len(df_all_team)} rows)')
else:
    team_files = sorted(glob.glob(os.path.join(dataset_dir, "game*_team.csv")), key=extract_game_num)
    if team_files:
        dfs = []
        for f in team_files:
            g = extract_game_num(f)
            d = pd.read_csv(f, index_col=False)
            d = d.loc[:, ~d.columns.str.contains('^Unnamed')]
            dfs.append(d)
        if dfs:
            df_all_team = pd.concat(dfs, ignore_index=True).reset_index(drop=True)
            print(f'Created df_all_team from CSV files ({len(df_all_team)} rows)')
        else:
            print('No valid team CSVs to aggregate.')
    else:
        print('No per-game team data available to aggregate.')

=== Menggabungkan Semua file per-game ke DataFrame (in-memory) ===
Created df_all_total from in-memory per_game_total (100 rows)
Created df_all_team from in-memory per_game_team (100 rows)


In [108]:
# Preview combined DataFrames (in-memory) if available
if 'df_all_total' in globals() and df_all_total is not None:
    print('=== PREVIEW df_all_total (Normalisasi Match) ===')
    display(df_all_total.head(20))
else:
    print('df_all_total not available in memory. Run aggregation cell first.')

if 'df_all_team' in globals() and df_all_team is not None:
    print('=== PREVIEW df_all_team (Normalisasi Team) ===')
    display(df_all_team.head(20))
else:
    print('df_all_team not available in memory. Run aggregation cell first.')

=== PREVIEW df_all_total (Normalisasi Match) ===


,K,D,A,Hero Dmg (%),Turret Dmg (%),Dmg Taken (%),TFP (%),Skor
0,7,5,5,8.26,33.36,9.71,52,7.8
1,2,6,6,7.19,9.14,4.57,35,4.8
2,0,4,12,4.63,3.63,7.29,52,6.9
3,7,7,4,9.73,29.62,35.31,48,6.0
4,10,5,5,11.08,2.91,13.99,65,9.3
5,1,3,11,8.02,9.66,2.51,44,7.1
6,7,2,6,15.92,4.01,3.43,48,8.4
7,6,6,5,13.55,7.66,9.43,41,6.7
8,3,6,15,4.75,0.00,8.46,67,8.0
9,10,6,8,16.88,0.00,5.30,67,8.5


=== PREVIEW df_all_team (Normalisasi Team) ===


,K,D,A,Hero Dmg (%),Turret Dmg (%),Dmg Taken (%),TFP (%),Skor
0,7,5,5,20.19,42.40,13.70,52,7.8
1,2,6,6,17.58,11.62,6.45,35,4.8
2,0,4,12,11.31,4.62,10.29,52,6.9
3,7,7,4,23.80,37.66,49.82,48,6.0
4,10,5,5,27.11,3.70,19.74,65,9.3
5,1,3,11,13.57,45.29,8.62,44,7.1
6,7,2,6,26.93,18.80,11.77,48,8.4
7,6,6,5,22.92,35.91,32.36,41,6.7
8,3,6,15,8.03,0.00,29.05,67,8.0
9,10,6,8,28.55,0.00,18.19,67,8.5


In [109]:
# Ensure df_all_total and df_all_team exist; attempt to load from CSV as fallback
if 'df_all_total' not in globals() or df_all_total is None:
    try:
        df_all_total = pd.read_csv(os.path.join(dataset_dir, "all_games_total.csv"))
        print('Loaded df_all_total from CSV fallback.')
    except Exception as e:
        print('df_all_total not in memory and failed to load from CSV:', e)

if 'df_all_team' not in globals() or df_all_team is None:
    try:
        df_all_team = pd.read_csv(os.path.join(dataset_dir, "all_games_team.csv"))
        print('Loaded df_all_team from CSV fallback.')
    except Exception as e:
        print('df_all_team not in memory and failed to load from CSV:', e)


In [110]:
df_all_total.info()
df_all_total.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   K               100 non-null    int64  
 1   D               100 non-null    int64  
 2   A               100 non-null    int64  
 3   Hero Dmg (%)    100 non-null    float64
 4   Turret Dmg (%)  100 non-null    float64
 5   Dmg Taken (%)   100 non-null    float64
 6   TFP (%)         100 non-null    int64  
 7   Skor            100 non-null    float64
dtypes: float64(4), int64(4)
memory usage: 6.4 KB


,K,D,A,Hero Dmg (%),Turret Dmg (%),Dmg Taken (%),TFP (%),Skor
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.0000
mean,5.770000,5.790000,9.390000,10.000000,9.999900,10.000000,52.920000,7.6540
std,4.840601,2.679119,5.355814,4.372223,11.697547,5.323339,15.946045,2.3708
min,0.000000,0.000000,2.000000,2.500000,0.000000,2.510000,16.000000,3.0000
25%,2.000000,4.000000,5.750000,6.750000,0.785000,6.492500,42.000000,5.9750
50%,5.000000,6.000000,8.000000,9.380000,5.450000,8.825000,52.000000,7.8000
75%,8.000000,7.250000,12.000000,12.682500,14.932500,12.432500,64.000000,9.0000
max,21.000000,13.000000,26.000000,23.480000,41.780000,35.310000,93.000000,15.4000


In [111]:
df_all_team.info()
df_all_team.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   K               100 non-null    int64  
 1   D               100 non-null    int64  
 2   A               100 non-null    int64  
 3   Hero Dmg (%)    100 non-null    float64
 4   Turret Dmg (%)  100 non-null    float64
 5   Dmg Taken (%)   100 non-null    float64
 6   TFP (%)         100 non-null    int64  
 7   Skor            100 non-null    float64
dtypes: float64(4), int64(4)
memory usage: 6.4 KB


,K,D,A,Hero Dmg (%),Turret Dmg (%),Dmg Taken (%),TFP (%),Skor
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.0000
mean,5.770000,5.790000,9.390000,20.000100,20.000400,19.999900,52.920000,7.6540
std,4.840601,2.679119,5.355814,8.235913,22.117838,9.339859,15.946045,2.3708
min,0.000000,0.000000,2.000000,4.850000,0.000000,6.450000,16.000000,3.0000
25%,2.000000,4.000000,5.750000,13.555000,1.772500,13.032500,42.000000,5.9750
50%,5.000000,6.000000,8.000000,18.930000,11.695000,18.840000,52.000000,7.8000
75%,8.000000,7.250000,12.000000,25.215000,35.287500,25.660000,64.000000,9.0000
max,21.000000,13.000000,26.000000,44.810000,97.210000,49.820000,93.000000,15.4000


Cek Korelasi

## regresi linear untuk skor yang persentasenya berdasarkan perteam

In [114]:
from sklearn.model_selection import train_test_split

# Pastikan kolom index-like tidak ikut sebagai fitur
x_team = df_all_team.drop(columns=['Skor'], errors='ignore')
y_team = df_all_team['Skor']

x_train, x_test, y_train, y_test = train_test_split(x_team, y_team, test_size=0.2, random_state=42)

print('Features used for training:', list(x_train.columns))

Features used for training: ['K', 'D', 'A', 'Hero Dmg (%)', 'Turret Dmg (%)', 'Dmg Taken (%)', 'TFP (%)']


In [115]:
# Gunakan hanya fitur numerik (abaikan kolom index yang tersisa)
X_df = x_train.select_dtypes(include=[np.number]).copy()
dropped = [c for c in x_train.columns if c not in X_df.columns]
if dropped:
    print('Dropped non-numeric columns from features:', dropped)
X = X_df.values
# Pastikan y numeric
y = pd.to_numeric(y_train, errors='coerce').fillna(0).values.reshape(-1,1)
# Tambah bias (intercept) kolom
X = np.column_stack((np.ones(len(X)), X))

XtX = X.T @ X
Xty = X.T @ y

print(XtX)
print(Xty)

[[8.00000000e+01 3.98000000e+02 4.59000000e+02 7.81000000e+02
  1.48411000e+03 1.42846000e+03 1.60281000e+03 4.11300000e+03]
 [3.98000000e+02 3.30600000e+03 2.13100000e+03 3.45200000e+03
  8.80574000e+03 8.31295000e+03 7.59785000e+03 2.23070000e+04]
 [4.59000000e+02 2.13100000e+03 3.24300000e+03 4.21700000e+03
  8.46972000e+03 8.13276000e+03 9.74243000e+03 2.21230000e+04]
 [7.81000000e+02 3.45200000e+03 4.21700000e+03 1.00430000e+04
  1.35656500e+04 1.18729100e+04 1.58238100e+04 4.36710000e+04]
 [1.48411000e+03 8.80574000e+03 8.46972000e+03 1.35656500e+04
  3.17888371e+04 3.05084223e+04 2.90297270e+04 7.89250000e+04]
 [1.42846000e+03 8.31295000e+03 8.13276000e+03 1.18729100e+04
  3.05084223e+04 5.80433622e+04 2.79385426e+04 7.28328800e+04]
 [1.60281000e+03 7.59785000e+03 9.74243000e+03 1.58238100e+04
  2.90297270e+04 2.79385426e+04 3.95793793e+04 8.19825600e+04]
 [4.11300000e+03 2.23070000e+04 2.21230000e+04 4.36710000e+04
  7.89250000e+04 7.28328800e+04 8.19825600e+04 2.31677000e+05]]

In [116]:
aug = np.hstack((XtX, Xty))
pd.DataFrame(aug)

,0,1,2,3,4,5,6,7,8
0,80.00,398.00,459.00,781.00,1484.1100,1428.4600,1602.8100,4113.00,593.000
1,398.00,3306.00,2131.00,3452.00,8805.7400,8312.9500,7597.8500,22307.00,3349.400
2,459.00,2131.00,3243.00,4217.00,8469.7200,8132.7600,9742.4300,22123.00,3060.800
3,781.00,3452.00,4217.00,10043.00,13565.6500,11872.9100,15823.8100,43671.00,6303.600
4,1484.11,8805.74,8469.72,13565.65,31788.8371,30508.4223,29029.7270,78925.00,11405.737
5,1428.46,8312.95,8132.76,11872.91,30508.4223,58043.3622,27938.5426,72832.88,10818.272
6,1602.81,7597.85,9742.43,15823.81,29029.7270,27938.5426,39579.3793,81982.56,11768.441
7,4113.00,22307.00,22123.00,43671.00,78925.0000,72832.8800,81982.5600,231677.00,32869.700


In [117]:
def gauss_jordan(A):
    A = A.astype(float)
    n = A.shape[0]

    for i in range(n):

        # pivot
        A[i] = A[i] / A[i, i]

        # eliminasi baris lain
        for j in range(n):
            if j != i:
                A[j] = A[j] - A[j, i] * A[i]

    return A

rref = gauss_jordan(aug.copy())

print(rref)

[[ 1.          0.          0.          0.          0.          0.
   0.          0.          3.57295491]
 [ 0.          1.          0.          0.          0.          0.
   0.          0.          0.24266832]
 [ 0.          0.          1.          0.          0.          0.
   0.          0.         -0.35265112]
 [ 0.          0.          0.          1.          0.          0.
   0.          0.          0.17750456]
 [ 0.          0.          0.          0.          1.          0.
   0.          0.          0.02345965]
 [ 0.          0.          0.          0.          0.          1.
   0.          0.          0.00698052]
 [ 0.          0.          0.          0.          0.          0.
   1.          0.          0.02370602]
 [ 0.          0.          0.          0.          0.          0.
   0.          1.          0.0367209 ]]


In [118]:
beta= rref[:, -1]
print("Koefisien beta:\n", beta)

Koefisien beta:
 [ 3.57295491  0.24266832 -0.35265112  0.17750456  0.02345965  0.00698052
  0.02370602  0.0367209 ]


In [119]:
# Gunakan koefisien beta untuk membuat prediksi pada set test dan bandingkan dengan y_test
# Siapkan fitur numerik pada x_test agar sesuai urutan kolom yang dipakai saat training (X_df)
train_numeric_cols = list(X_df.columns)  # kolom numerik yang digunakan saat training
X_test_df = x_test.select_dtypes(include=[np.number]).copy()
# Pastikan semua kolom training ada di test; isi dengan 0 jika tidak ada
for c in train_numeric_cols:
    if c not in X_test_df.columns:
        X_test_df[c] = 0
# Seleksi dan urutkan kolom sesuai training
X_test_df = X_test_df[train_numeric_cols].fillna(0)
# Tambah kolom bias (intercept)
X_test_mat = np.column_stack((np.ones(len(X_test_df)), X_test_df.values))
# Pastikan bentuk beta kompatibel (vektor kolom)
beta_vec = np.array(beta).reshape(-1)
if beta_vec.shape[0] != X_test_mat.shape[1]:
    raise ValueError(f'Ukuran beta ({beta_vec.shape[0]}) tidak cocok dengan jumlah fitur+intercept ({X_test_mat.shape[1]}).')
# Hitung prediksi
y_pred = (X_test_mat @ beta_vec).flatten()
# Ambil y_test numerik yang sesuai
y_test_num = pd.to_numeric(y_test, errors='coerce').fillna(0).values.flatten()
# Buat DataFrame perbandingan
cmp = pd.DataFrame({'y_true': y_test_num, 'y_pred': np.round(y_pred,2)})
cmp['diff'] = np.round(cmp['y_true'] - cmp['y_pred'],2)
display(cmp.reset_index(drop=True))
# Hitung metrik sederhana: MSE, MAE, R2 (manual)
ss_res = np.sum((y_test_num - y_pred)**2)
ss_tot = np.sum((y_test_num - y_test_num.mean())**2)
r2 = 1 - ss_res/ss_tot if ss_tot != 0 else np.nan
mse = np.mean((y_test_num - y_pred)**2)
mae = np.mean(np.abs(y_test_num - y_pred))
print(f'R2: {r2:.4f} | MSE: {mse:.4f} | MAE: {mae:.4f}')

,y_true,y_pred,diff
0,11.3,10.70,0.60
1,8.9,9.36,-0.46
2,7.7,8.03,-0.33
3,7.2,7.15,0.05
4,9.0,8.43,0.57
5,7.0,7.05,-0.05
6,7.3,7.61,-0.31
7,6.4,6.47,-0.07
8,12.9,12.35,0.55
9,7.8,7.40,0.40


R2: 0.9766 | MSE: 0.1435 | MAE: 0.3133


## regresi linear untuk skor yang persentasenya berdasarkan keseluruhan pemain dalam 1 match

In [120]:
# Pastikan kolom index-like tidak ikut sebagai fitur
x_total = df_all_total.drop(columns=['Skor'], errors='ignore')
y_total = df_all_total['Skor']

x_train, x_test, y_train, y_test = train_test_split(x_total, y_total, test_size=0.2, random_state=42)

print('Features used for training:', list(x_train.columns))

Features used for training: ['K', 'D', 'A', 'Hero Dmg (%)', 'Turret Dmg (%)', 'Dmg Taken (%)', 'TFP (%)']


In [121]:
# Gunakan hanya fitur numerik (abaikan kolom index yang tersisa)
X_df = x_train.select_dtypes(include=[np.number]).copy()
dropped = [c for c in x_train.columns if c not in X_df.columns]
if dropped:
    print('Dropped non-numeric columns from features:', dropped)
X = X_df.values
# Pastikan y numeric
y = pd.to_numeric(y_train, errors='coerce').fillna(0).values.reshape(-1,1)
# Tambah bias (intercept) kolom
X = np.column_stack((np.ones(len(X)), X))

XtX = X.T @ X
Xty = X.T @ y

print(XtX)
print(Xty)

[[8.00000000e+01 3.98000000e+02 4.59000000e+02 7.81000000e+02
  7.43790000e+02 6.91520000e+02 7.95300000e+02 4.11300000e+03]
 [3.98000000e+02 3.30600000e+03 2.13100000e+03 3.45200000e+03
  4.46498000e+03 4.28174000e+03 3.69995000e+03 2.23070000e+04]
 [4.59000000e+02 2.13100000e+03 3.24300000e+03 4.21700000e+03
  4.18987000e+03 3.87465000e+03 4.90897000e+03 2.21230000e+04]
 [7.81000000e+02 3.45200000e+03 4.21700000e+03 1.00430000e+04
  6.89976000e+03 6.08098000e+03 7.69674000e+03 4.36710000e+04]
 [7.43790000e+02 4.46498000e+03 4.18987000e+03 6.89976000e+03
  8.08916170e+03 7.39077760e+03 7.12481390e+03 3.96480900e+04]
 [6.91520000e+02 4.28174000e+03 3.87465000e+03 6.08098000e+03
  7.39077760e+03 1.44938136e+04 6.75810620e+03 3.49163200e+04]
 [7.95300000e+02 3.69995000e+03 4.90897000e+03 7.69674000e+03
  7.12481390e+03 6.75810620e+03 1.04057284e+04 4.04918800e+04]
 [4.11300000e+03 2.23070000e+04 2.21230000e+04 4.36710000e+04
  3.96480900e+04 3.49163200e+04 4.04918800e+04 2.31677000e+05]]

In [122]:
aug = np.hstack((XtX, Xty))
pd.DataFrame(aug)

,0,1,2,3,4,5,6,7,8
0,80.00,398.00,459.00,781.00,743.7900,691.5200,795.3000,4113.00,593.000
1,398.00,3306.00,2131.00,3452.00,4464.9800,4281.7400,3699.9500,22307.00,3349.400
2,459.00,2131.00,3243.00,4217.00,4189.8700,3874.6500,4908.9700,22123.00,3060.800
3,781.00,3452.00,4217.00,10043.00,6899.7600,6080.9800,7696.7400,43671.00,6303.600
4,743.79,4464.98,4189.87,6899.76,8089.1617,7390.7776,7124.8139,39648.09,5773.581
5,691.52,4281.74,3874.65,6080.98,7390.7776,14493.8136,6758.1062,34916.32,5360.940
6,795.30,3699.95,4908.97,7696.74,7124.8139,6758.1062,10405.7284,40491.88,5763.933
7,4113.00,22307.00,22123.00,43671.00,39648.0900,34916.3200,40491.8800,231677.00,32869.700


In [123]:
def gauss_jordan(A):
    A = A.astype(float)
    n = A.shape[0]

    for i in range(n):

        # pivot
        A[i] = A[i] / A[i, i]

        # eliminasi baris lain
        for j in range(n):
            if j != i:
                A[j] = A[j] - A[j, i] * A[i]

    return A

rref = gauss_jordan(aug.copy())

print(rref)

[[ 1.          0.          0.          0.          0.          0.
   0.          0.          3.61036389]
 [ 0.          1.          0.          0.          0.          0.
   0.          0.          0.23320011]
 [ 0.          0.          1.          0.          0.          0.
   0.          0.         -0.34559056]
 [ 0.          0.          0.          1.          0.          0.
   0.          0.          0.17249809]
 [ 0.          0.          0.          0.          1.          0.
   0.          0.          0.05217236]
 [ 0.          0.          0.          0.          0.          1.
   0.          0.          0.01186039]
 [ 0.          0.          0.          0.          0.          0.
   1.          0.          0.03625009]
 [ 0.          0.          0.          0.          0.          0.
   0.          1.          0.0387614 ]]


In [124]:
beta= rref[:, -1]
print("Koefisien beta:\n", beta)

Koefisien beta:
 [ 3.61036389  0.23320011 -0.34559056  0.17249809  0.05217236  0.01186039
  0.03625009  0.0387614 ]


In [125]:
# Gunakan koefisien beta untuk membuat prediksi pada set test dan bandingkan dengan y_test
# Siapkan fitur numerik pada x_test agar sesuai urutan kolom yang dipakai saat training (X_df)
train_numeric_cols = list(X_df.columns)  # kolom numerik yang digunakan saat training
X_test_df = x_test.select_dtypes(include=[np.number]).copy()
# Pastikan semua kolom training ada di test; isi dengan 0 jika tidak ada
for c in train_numeric_cols:
    if c not in X_test_df.columns:
        X_test_df[c] = 0
# Seleksi dan urutkan kolom sesuai training
X_test_df = X_test_df[train_numeric_cols].fillna(0)
# Tambah kolom bias (intercept)
X_test_mat = np.column_stack((np.ones(len(X_test_df)), X_test_df.values))
# Pastikan bentuk beta kompatibel (vektor kolom)
beta_vec = np.array(beta).reshape(-1)
if beta_vec.shape[0] != X_test_mat.shape[1]:
    raise ValueError(f'Ukuran beta ({beta_vec.shape[0]}) tidak cocok dengan jumlah fitur+intercept ({X_test_mat.shape[1]}).')
# Hitung prediksi
y_pred = (X_test_mat @ beta_vec).flatten()
# Ambil y_test numerik yang sesuai
y_test_num = pd.to_numeric(y_test, errors='coerce').fillna(0).values.flatten()
# Buat DataFrame perbandingan
cmp = pd.DataFrame({'y_true': y_test_num, 'y_pred': np.round(y_pred,2)})
cmp['diff'] = np.round(cmp['y_true'] - cmp['y_pred'],2)
display(cmp.reset_index(drop=True))
# Hitung metrik sederhana: MSE, MAE, R2 (manual)
ss_res = np.sum((y_test_num - y_pred)**2)
ss_tot = np.sum((y_test_num - y_test_num.mean())**2)
r2 = 1 - ss_res/ss_tot if ss_tot != 0 else np.nan
mse = np.mean((y_test_num - y_pred)**2)
mae = np.mean(np.abs(y_test_num - y_pred))
print(f'R2: {r2:.4f} | MSE: {mse:.4f} | MAE: {mae:.4f}')

,y_true,y_pred,diff
0,11.3,10.58,0.72
1,8.9,9.26,-0.36
2,7.7,8.27,-0.57
3,7.2,7.23,-0.03
4,9.0,8.38,0.62
5,7.0,7.16,-0.16
6,7.3,7.60,-0.30
7,6.4,6.54,-0.14
8,12.9,12.46,0.44
9,7.8,7.57,0.23


R2: 0.9773 | MSE: 0.1390 | MAE: 0.3210
